In [ ]:
# current wip
# encounter level tab data
# monthly aggregates of labs and aki counts on the encounter level

In [ ]:
# %%
# polars script with datatype toggles
# read_csv
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import os
import re
import logging
from tqdm import tqdm
 

# %%
# variables
output_fname = "processed_tab_eskd_v5.csv"
icd_file = "/opt/data/commonfilesharePHI/ldiao/ckd_project/icd_mapping.csv"
subset = True# <<

if subset: 
    subset_size = "10000"  # 10, 100, full # <<
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}"
    # output_dir = f"/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_subset_{subset_size}"
    event_file =  f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}/unprocessed_tab_subset_{subset_size}.csv"
if not subset:
    # output_dir = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_full"
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_full"
    # event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"
    event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v3.rpt.parquet"


try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
except FileExistsError:
    print(f"Output directory already exists: {output_dir}")

print(f"Processing started. Output directory: {output_dir}")

# %%
# Setup logging
log_file_path = os.path.join(output_dir, "tab_gen_m.log")
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"Processing started. Output directory: {output_dir}")

# %%
# -----------------------------
# Load and preprocess with Polars
# -----------------------------
# Using pl.read_csv to load the entire file into a DataFrame
# Add a toggle to switch between separators
print(event_file)
if subset:
    df = pl.read_csv(
        event_file,
        separator='$',
        infer_schema_length=None,
        null_values=["null", "NULL"],
    ).unique()

    df = df.with_columns(
        pl.col("PatientID").cast(pl.Utf8, strict=False),
        pl.col("EventTimeStamp").cast(pl.Utf8,  strict=False),
        pl.col("DataCategory").cast(pl.Utf8, strict=False),
        pl.col("DataType").cast(pl.Utf8, strict=False),
        pl.col("DataNumeric").cast(pl.Float64, strict=False),
    )
if not subset: 
    csv = event_file 
    df = pl.read_parquet(csv)
    print(len(df['PatientID'].unique()))

logger.info(f"Initial DataFrame schema: {df.schema}")

In [ ]:
df.dtypes

In [ ]:
print(df.shape)

In [ ]:
# -----------------------------
# ICD code -> long title mapping (ported from embedding_gen_pl_v3.py)
# -----------------------------
icd_map_df = pl.read_csv(icd_file)
icd_map_df = icd_map_df.with_columns(
    pl.col("icd_code").cast(pl.Utf8).str.replace(".", "", literal=True)
)
icd_map = dict(zip(icd_map_df["icd_code"], icd_map_df["long_title"]))
logger.info(f"Loaded {len(icd_map)} ICD code -> long_title mappings from {icd_file}")


In [ ]:
icd_map_df.shape

In [ ]:
len(icd_map.keys())

In [ ]:
df.shape

In [ ]:
# Extract and forward-fill ICD

custom_map = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,  # ESRD
    9: 0,  # CKD, unspecified stage
}


In [ ]:
print("Building Filter")
# ckd_icd_df's only job now is the patient-level cohort filter (max_stage >= 3).
# Day-level ICD info (icd_daywise, below) is derived from df directly instead,
# since df already has EventDate computed and doesn't need re-plumbing here.
ckd_icd_df = (
    df.filter(pl.col("DataCategory").str.contains("N18"))
    .with_columns(
      pl.col("DataCategory")
        .str.extract(r"N18\.([1-9])", 1)
        .cast(pl.Int64)
        .replace(custom_map, default=None)
        .alias("CKD_stage_numeric")
    )
    .select([pl.col("PatientID"), pl.col("EventTimeStamp"), pl.col("META_1"), pl.col("CKD_stage_numeric")])
    .with_columns(
        pl.col("CKD_stage_numeric")
            .max()
            .over("PatientID")
            .alias("max_stage")
    )
    .filter(pl.col("max_stage") >= 3)
    .unique()
) 

In [ ]:
ckd_icd_df.shape

In [ ]:
df

In [ ]:

print("Filtering and converting")
df = (
    df
    .join(
        ckd_icd_df.select("PatientID").unique(), 
        on="PatientID", 
        how='inner')
    .drop_nulls(subset=["DataNumeric"])
    .with_columns([
        pl.col("EventTimeStamp").str.strptime(pl.Datetime("us")),
        pl.col("DataCategory").fill_null(pl.col("META_2"))
    ])
    .with_columns(
        pl.col("EventTimeStamp").dt.date().alias("EventDate")
    )

)
print(len(df['PatientID'].unique()))
# %% demographics and encounter


In [ ]:
df.shape

In [ ]:
# Encounter-level skeleton: one row per (PatientID, META_1) encounter.
# icd_daywise is gone -- ckd_icd_df already exists (cell 10) and carries the
# ICD/CKD-stage info we need; we join it in at the end, at encounter grain.
# EventMonth (year-month of the encounter's first date) is the join key used
# below to broadcast monthly-aggregated features (labs, AKI) onto each encounter.

enc_grouped_df = (
    df.group_by(["PatientID", "META_1"])
    .agg(pl.col("EventDate").min().alias("EventDate"))
    .with_columns(
        pl.col("EventDate").dt.strftime("%Y-%m").alias("EventMonth")
    )
    .sort(["PatientID", "META_1"])
)
enc_grouped_df.shape

In [ ]:
# enc_grouped_df = (
#     df.group_by(["PatientID", "META_1"])
#     .agg(pl.col("EventDate").min().alias("EventDate"))
#     .with_columns(
#         pl.col("EventDate").dt.strftime("%Y-%m").alias("EventMonth")
#     )
#     .sort(["PatientID", "META_1"])
# )
# enc_grouped_df.shape

In [ ]:
enc_grouped_df.head

In [ ]:
# %%
aki_icd_codes = ["N17.0", "N17.1", "N17.2", "N17.8", "N17.9"]
aki_events = df.filter(
    (pl.col("DataType") == "Diagnosis") &
    (pl.col("DataCategory").is_in(aki_icd_codes))
).with_columns(
    pl.col("EventDate").dt.strftime("%Y-%m").alias("EventMonth")
)

# Monthly AKI count (ICD-code based), keyed on PatientID + EventMonth so it can
# be broadcast to every encounter that falls in that patient-month.
aki_count = aki_events.group_by(["PatientID", "EventMonth"]).agg([
    pl.len().alias("AKI_ICD_Total"),
])

In [ ]:
print(enc_grouped_df.schema)
print(aki_count.schema)  # and lab_pivot.schema

In [ ]:
# Join monthly AKI counts onto the encounter-level skeleton via PatientID + EventMonth.
# Every encounter in a given patient-month gets that month's AKI count.
enc_grouped_df = enc_grouped_df.join(
    aki_count, on=["PatientID", "EventMonth"], how="left", join_nulls=True
).with_columns(
    pl.col("AKI_ICD_Total").fill_null(0)  # Ensure months with no AKI are 0, not null
)

enc_grouped_df.head()

In [ ]:
# %%
# top lab features from the paper ---
top_lab_features = [
    "CREATININE", "GFR", "GFREST", "ALBUMIN/CREATININE RATIO",
    "PROTEIN/CREATININE RATIO", "BUN", "PTH"
]

lab_df = df.filter(
    (pl.col("DataType") == "Labs") &
    (pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().str.contains("|".join(top_lab_features)))
).with_columns([
    pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().alias("LabCategory"),
    pl.col("EventDate").dt.strftime("%Y-%m").alias("EventMonth"),
])

# Monthly aggregates per lab: mean, min, max, std, count -- one row per
# PatientID + EventMonth + LabCategory.
lab_monthly = lab_df.group_by(["PatientID", "EventMonth", "LabCategory"]).agg([
    pl.col("DataNumeric").mean().alias("mean"),
    pl.col("DataNumeric").min().alias("min"),
    pl.col("DataNumeric").max().alias("max"),
    pl.col("DataNumeric").std().alias("std"),
    pl.col("DataNumeric").count().alias("count"),
])

# %%
lab_monthly

# %%
# Pivot wide: one column per (stat, LabCategory) combo, e.g. "mean_CREATININE".
lab_pivot = lab_monthly.pivot(
    index=["PatientID", "EventMonth"],
    on="LabCategory",
    values=["mean", "min", "max", "std", "count"],
)

# %%
lab_pivot

In [ ]:
print(enc_grouped_df.shape)

In [ ]:
# %%
# Prefix all pivoted lab-stat columns with "lab_" for clarity downstream.
rename_dict = {c: f"lab_{c}" for c in lab_pivot.columns if c not in ("PatientID", "EventMonth")}
lab_pivot = lab_pivot.rename(rename_dict)

enc_grouped_df = enc_grouped_df.join(lab_pivot, on=["PatientID", "EventMonth"], how="left", join_nulls=True)

In [ ]:
# Merge in ICD/CKD-stage info from ckd_icd_df (matches the pl notebook's join pattern).
# ckd_icd_df has one row per N18 diagnosis event, so it must be collapsed to one
# row per (PatientID, META_1) first -- otherwise this join fans out and duplicates
# encounter rows.
enc_grouped_df = enc_grouped_df.join(
    ckd_icd_df.select(["PatientID", "META_1", "CKD_stage_numeric", "max_stage"])
    .group_by(["PatientID", "META_1"])
    .agg([
        pl.col("CKD_stage_numeric").max().alias("CKD_stage_numeric"),
        pl.col("max_stage").max().alias("max_stage"),
    ]),
    on=["PatientID", "META_1"], how='left'
)

In [ ]:
print(enc_grouped_df.shape)
base_df = enc_grouped_df

In [ ]:
# -----------------------------
# Final report
# -----------------------------
base_df = enc_grouped_df  # keep the rest of this cell unchanged below

logger.info(f"[INFO] Final tabular shape: {base_df.shape}")
logger.info(f"[INFO] Sample features:\n{base_df.head()}")
logger.info(f"[INFO] CKD stage counts:\n{base_df['CKD_stage_numeric'].value_counts(sort=True)}")
base_df_path = os.path.join(output_dir, output_fname)
logger.info(f"Writing final DataFrame of shape {base_df.shape} to {base_df_path}")
base_df.write_csv(base_df_path)

logger.info("End of Tabular Generation")

# check csv
# Construct the full file path
final_file_path = os.path.join(output_dir, output_fname)
print(final_file_path)
# Read the processed CSV file
try:
    final_df = pl.read_csv(final_file_path)
    print("File read successfully.")
    print(final_df.head())
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

In [ ]:
# %%
import polars as pl
tab_path = f"./tabular_full/{output_fname}"    
df = pl.read_csv(tab_path)

# Overall row/patient counts
print(f"Rows: {df.shape[0]}, Patients: {df['PatientID'].n_unique()}")

# Prevalence of CKD_stage (row-level and patient-level, since patients can span stages)
print(df["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / df.shape[0] * 100).round(2).alias("pct_rows")
))

# Patient-level prevalence: each patient's max (worst) CKD_stage
patient_stage = df.group_by("PatientID").agg(pl.col("CKD_stage_numeric").max())
print(patient_stage["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / patient_stage.shape[0] * 100).round(2).alias("pct_patients")
))



In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df

In [ ]:
len(df['PatientID'].unique())

In [ ]:
prediction_period = 365 # 365, 730, 1095
years = str(round(prediction_period/365))
exclude_cols = ['PatientID', 
                    'EventDate', 
                    'EventMonth',
                    'CKD_stage', # Raw stage column
                    'CKD_stage_clean', # Intermediate cleaned stage
                    'CKD_stage_numeric',
                    # 'CKD_stage_numeric_right'
                    'max_stage',
                    # 'max_stage_right'
                    "META_1",
                    'label_ckd_stage_4_plus', f'label_ckd_{years}_year_future', # Generated labels
                    'time_until_progression', 'event_for_cox_indicator'] # Generated TTE info
    

In [ ]:
df.columns

In [ ]:
list(set(df.columns) - set(exclude_cols))